# Importing libraries and data

In [ ]:
import foldersfiles as ff
import functions as fn

import numpy as np
import pandas as pd

import pickle

In [ ]:
!ls "{ff.CLEANED_ORIGINALS_DATA_DIR / ff.PARQUETS}"

In [ ]:
!ls "{ff.CLEANED_PROCESSED_DATA_DIR / ff.PICKLES}"

In [ ]:
df = fn.import_func(
    ff.CLEANED_ORIGINALS_DATA_DIR / ff.PARQUETS / f'{ff.ICONS_RICOV}.parquet'
)

# Overview

In [ ]:
df.info()

In [ ]:
df

# Analysis

## Supposed hospitalizations periods

In [ ]:
df_dates = df[['pseudopaz', 'data_apertura', 'data_chiusura', 'data_chiusura_sdo']]
df_dates = df_dates.sort_values(['pseudopaz']).reset_index(drop=True)
df_dates

### Relationship between `data_chiusura` and `data_chiusura_sdo`

In [ ]:
# Check the difference between the 'data_chiusura' and the 'data_chiusura_sdo' fields (NaTs are ignored)

df_dates['diff_data_chiusura'] = df_dates['data_chiusura_sdo'] - df_dates['data_chiusura']
(
    df_dates[['diff_data_chiusura']]
    .dropna()
    .sort_values(['diff_data_chiusura'])
    .reset_index(drop=True)
)
# df_dates['diff_data_chiusura'].describe()

`diff_data_chiusura = data_chiusura_sdo - data_chiusura`. Positive values mean
that the SDO closing date is later; negative values mean it is earlier. Zero
means the timestamps match exactly. Convert timedeltas to **fractional days**
with `total_seconds()` so hours and minutes are preserved, including for
negative differences. Missing differences are excluded and counted explicitly.
Each row contributes once; repeated records and multiple stays for a patient
therefore retain their existing weight.

The plots below complement one another:

- **Full-range histogram:** shows the shape and extreme differences. Bin choices
  affect its appearance, and a long tail can compress the main cluster.
- **Central-range histogram (1st–99th percentiles):** reveals the main cluster.
  Its title reports excluded observations; use the full-range plot for tails.

In [ ]:
import matplotlib.pyplot as plt

# Keep the sign and sub-day precision; do not use .dt.days (which floors values).
diff_days = df_dates['diff_data_chiusura'].dt.total_seconds().div(86_400)
observed_days = diff_days.dropna()
n_missing = int(diff_days.isna().sum())
print(f"Observed differences: {len(observed_days):,}; missing: {n_missing:,}")

if observed_days.empty:
    print("No observed closing-date differences to plot.")
else:
    print(
        f"Negative: {(observed_days < 0).sum():,}; "
        f"zero: {(observed_days == 0).sum():,}; "
        f"positive: {(observed_days > 0).sum():,}"
    )
    median_days = observed_days.median()
    lower, upper = observed_days.quantile([0.01, 0.99])
    central_days = observed_days[observed_days.between(lower, upper)]
    n_outside = len(observed_days) - len(central_days)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), layout='constrained')
    for ax, values, title in [
        (axes[0], observed_days, 'Full distribution'),
        (axes[1], central_days,
         f'1st–99th percentile range\n{n_outside:,} observations outside range'),
    ]:
        ax.hist(values, bins=50, color='steelblue', edgecolor='white')
        ax.axvline(0, color='black', linestyle='--', linewidth=1, label='Zero')
        ax.axvline(median_days, color='darkorange', linewidth=1.5,
                   label=f'Overall median: {median_days:.2f} days')
        ax.set(title=title, xlabel='Difference (days)', ylabel='Number of rows')
        ax.legend(fontsize='small')

    for ax in axes:
        ax.grid(axis='y', alpha=0.25)
        ax.set_axisbelow(True)
    fig.suptitle(
        'Closing-date difference: data_chiusura_sdo − data_chiusura\n'
        f'{len(observed_days):,} observed rows; {n_missing:,} missing',
        fontsize=13,
    )
    plt.show()


**Observation**: `data_chiusura` is *earlier* than `data_chiusura_sdo` in all cases except 3 rows: 161, 1142, 1148

In [ ]:
# Resets df_dates to original state

df_dates = df[['pseudopaz', 'data_apertura', 'data_chiusura', 'data_chiusura_sdo']]
df_dates = df_dates.sort_values(['pseudopaz']).reset_index(drop=True)
df_dates

### Relationship between `data_apertura` and `data_chiusura`

Check that `data_apertura` is before `data_chiusura` for every row

In [ ]:
df_dates['check_date_order'] = df_dates['data_apertura'] < df_dates['data_chiusura']
print(df_dates['check_date_order'].value_counts())

# rows where the check fails
invalid_rows = df_dates[~df_dates['check_date_order']]
invalid_rows

Assuming that `data_apertura` and `data_chiusura` are the true columns to watch...

... this code generates an overview of the number of hospitalizations, the total and mean time spent in the hospital

In [ ]:
# 2) compute the duration of each stay
df_dates['durata'] = df_dates['data_chiusura'] - df_dates['data_apertura']

# aggregate per pseudopatient
summary = df_dates.groupby('pseudopaz')['durata'].agg(['count', 'sum', 'mean'])
summary.columns = ['n_ricoveri', 'tempo_totale', 'tempo_medio']

# round to the nearest day
summary['tempo_totale'] = summary['tempo_totale'].dt.round('D')
summary['tempo_medio'] = summary['tempo_medio'].dt.round('D')

summary = summary.reset_index()

summary

## Supposed hospitalizations locations

In [ ]:
region_map = {
    '042001': 'trento',
    '042004': 'trento',
    '042005': 'trento',
    '042007': 'trento',
    '042014': 'trento',
    '042015': 'trento',
    '042022': 'trento',
    '042006': 'rovereto',
    '042010': 'rovereto',
    '042016': 'rovereto',
    '042018': 'rovereto',
    '042019': 'rovereto',
    '042020': 'rovereto',
}

In [ ]:
hospital_map = {
    '042001': 'tn schiara',
    '042004': 'b_valsugana',
    '042005': 'cles',
    '042006': 'rovereto',
    '042007': 'tione',
    '042010': 'arco',
    '042014': 'cavalese',
    '042015': 'tn scamillo',
    '042016': 's.stefano/s.pancrazio',
    '042018': 'cdc eremo_arco',
    '042019': 'cdc regina',
    '042020': 'cdc solatrix',
    '042022': 'cdc villa_bianca',
}

# hospital_map = {
#     '042001': 'OSPEDALE DI TRENTO',
#     '042004': 'PRESIDIO OSPEDALIERO DI BORGO VALSUGANA',
#     '042005': 'PRESIDIO OSPEDALIERO DI CLES',
#     '042006': 'OSPEDALE DI ROVERETO',
#     '042007': 'PRESIDIO OSPEDALIERO DI TIONE',
#     '042010': 'PRESIDIO OSPEDALIERO DI ARCO',
#     '042014': 'PRESIDIO OSPEDALIERO DI CAVALESE',
#     '042015': 'OSPEDALE SAN CAMILLO',
#     '042016': 'IST.RIAB.S.STEFANO-U.O. OSP. S.PANCRAZIO',
#     '042018': 'CASA DI CURA EREMO DI ARCO SRL',
#     '042019': 'CASA DI CURA REGINA',
#     '042020': 'CASA DI CURA SOLATRIX',
#     '042022': 'CASA DI CURA VILLA BIANCA',
# }

In [ ]:
df_strutture = df[['pseudopaz', 'struttura_cod']]

# df_strutture = df_strutture.sort_values(['pseudo_paz', 'data_apertura']).reset_index(drop=true)

In [ ]:
df_strutture['regione'] = df_strutture['struttura_cod'].map(region_map)

region_counts = df_strutture.groupby('pseudopaz')['regione'].nunique()

same_region = region_counts[region_counts == 1].index
different_region = region_counts[region_counts > 1].index

print('Patients always in same region:', len(same_region))
print('Patients across different regions:', len(different_region))

In [ ]:
df_strutture['ospedale'] = df_strutture['struttura_cod'].map(hospital_map)

switched_df_strutture = df_strutture[df_strutture['pseudopaz'].isin(different_region)]

regions = switched_df_strutture.groupby('pseudopaz').agg(
    codici_struttura=('struttura_cod', 'unique'),
    strutture_visitate=('ospedale', 'unique')
).reset_index()

regions.insert(1, 'n_strutture', regions['codici_struttura'].apply(len))

regions['codici_struttura'] = regions['codici_struttura'].apply(lambda x: ', '.join(x))
regions['strutture_visitate'] = regions['strutture_visitate'].apply(lambda x: ', '.join(x))

regions = regions.sort_values('n_strutture', ascending=False)

regions = regions.reset_index(drop=True)

regions

## Export

In [ ]:
# fn.export_func(df_, ff.CLEANED_PROCESSED_DATA_DIR, '', filetype=[
#     # 'parquet' ,
#     # 'pickle' ,
#     'excel'
# ])